In [2]:
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras import layers
from sklearn.preprocessing import normalize

In [3]:
#Dataset'i yükleyelim

X = np.load("../data/X.npy")
y_user = np.load("../data/y_user.npy")

print("X:",X.shape)
print("y_user:",y_user.shape)


X: (3521, 300, 90)
y_user: (3521,)


In [4]:
#User split yapalım

train_users=[0,1,2,3,6,7]
known_user=8
unknown_user=9

train_mask=np.isin(y_user,train_users)

X_train= X[train_mask]
y_train =y_user [train_mask]

print("Train:",X_train.shape)

Train: (2881, 300, 90)


In [5]:
#Train istatistikleri
mean = np.mean(X_train)
std = np.std(X_train)

X_train = (X_train-mean)/std

print("Normalization tamam.")

Normalization tamam.


In [6]:
#Activity modelini backbone olarak kullanacağız.

activity_model = load_model("../models/activity_model.h5")

#Softmax'tan önceki katman
base_model= Model(
    inputs=activity_model.layers[0].input,
    outputs=activity_model.layers[-2].output
)
base_model.summary()

Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 300, 90)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 300, 64)        │        28,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 300, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 150, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 150, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 150, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 75, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 75, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 75, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 186,816 (729.75 KB)

 Trainable params: 185,920 (726.25 KB)

 Non-trainable params: 896 (3.50 KB)

In [7]:
for layer in base_model.layers:
    layer.trainable = False

#Sadece son Dense katmanını aç
base_model.layers[-1].trainable = True

base_model.summary()

Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 300, 90)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 300, 64)        │        28,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 300, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 150, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 150, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 150, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 75, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 75, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 75, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 186,816 (729.75 KB)

 Trainable params: 32,896 (128.50 KB)

 Non-trainable params: 153,920 (601.25 KB)

In [8]:
#Triplet modeli kuralım

#Triplet inputs
input_anchor = layers.Input(shape=(300,90))
input_positive = layers.Input(shape=(300,90))
input_negative = layers.Input(shape=(300,90))

#Embedding üretelim
emb_anchor = base_model(input_anchor)
emb_positive = base_model(input_positive)
emb_negative = base_model(input_negative)

triplet_net=Model(
    inputs=[input_anchor,input_positive,input_negative],
    outputs =[emb_anchor,emb_positive,emb_negative]
)

print(type(triplet_net))



<class 'keras.src.models.functional.Functional'>


In [9]:
def generate_triplet_batch(X,y,batch_size=32):
    anchors=[]
    positives=[]
    negatives=[]
    
    unique_users = np.unique(y)
    
    for _ in range(batch_size):
        #Anchor user seç
        user = random.choice(unique_users)
        
        user_indices =np.where(y == user)[0]
        neg_user = random.choice(unique_users[unique_users != user])
        neg_indices = np.where(y == neg_user)[0]
        
        #Anchor&Positive
        a,p =np.random.choice(user_indices,2,replace=False)
        
        #Negative
        n = np.random.choice(neg_indices)
        
        anchors.append(X[a])
        positives.append(X[p])
        negatives.append(X[n])
        
    return np.array(anchors),np.array(positives),np.array(negatives)

In [10]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

In [11]:
#Training Step

def train_step(anchor, positive, negative):
    with tf.GradientTape() as tape:
        
        emb_anchor, emb_positive, emb_negative = triplet_net(
            [anchor, positive, negative]
        )
        
        loss = triplet_loss(emb_anchor, emb_positive, emb_negative)
    
    gradients = tape.gradient(loss, triplet_net.trainable_variables)
    optimizer.apply_gradients(zip(gradients, triplet_net.trainable_variables))
    
    return loss

In [12]:
def triplet_loss(anchor, positive, negative, margin=0.3):
    pos_dist = tf.reduce_sum(tf.square(anchor - positive), axis=1)
    neg_dist = tf.reduce_sum(tf.square(anchor - negative), axis=1)
    
    loss = tf.maximum(pos_dist - neg_dist + margin, 0.0)
    return tf.reduce_mean(loss)

In [13]:
a, p, n = generate_triplet_batch(X_train, y_train, 4)
loss = train_step(a, p, n)
print(loss)

tf.Tensor(29.60445, shape=(), dtype=float32)


In [14]:
#Training Loop

EPOCHS = 10
BATCH_SIZE = 32
STEPS_PER_EPOCH = 50

for epoch in range(EPOCHS):
    epoch_loss=0
    
    for step in range(STEPS_PER_EPOCH):
        a,p,n=generate_triplet_batch(X_train,y_train,BATCH_SIZE)
        
        loss = train_step(a, p, n)
        epoch_loss += loss.numpy()
        
    print(f"Epoch{epoch+1},Loss:{epoch_loss/STEPS_PER_EPOCH:.4f}")


Epoch1,Loss:42.7738
Epoch2,Loss:36.3920
Epoch3,Loss:31.7621
Epoch4,Loss:28.5554
Epoch5,Loss:27.2382
Epoch6,Loss:19.1303
Epoch7,Loss:17.6699
Epoch8,Loss:15.9770
Epoch9,Loss:15.5070
Epoch10,Loss:11.3535


In [15]:
#Train embeddings
train_embeddings =base_model.predict(X_train,verbose=0)

#Normalize
train_embeddings =normalize(train_embeddings,axis=1)


In [16]:
gallery ={}

for user in train_users:
    user_mask = (y_train == user)
    user_embeddings = train_embeddings[user_mask]
    
    mean_embedding = np.mean (user_embeddings,axis=0)
    mean_embedding =mean_embedding/np.linalg.norm(mean_embedding)
    
    gallery[user]=mean_embedding

print("Gallery hazır.")

Gallery hazır.


In [17]:
X_known = (X[np.isin(y_user,[8])]-mean)/std
X_unknown = (X[np.isin(y_user,[9])]-mean)/std

known_embeddings = base_model.predict(X_known,verbose=0)
unknown_embeddings = base_model.predict(X_unknown,verbose=0)

known_embeddings = normalize(known_embeddings,axis=1)
unknown_embeddings = normalize(unknown_embeddings,axis=1)


In [18]:
def cosine_similarity(a,b):
    return np.dot(a,b)

#Known accuracy
correct=0
for emb in known_embeddings:
    sims= {user:cosine_similarity(emb,gallery[user])for user in gallery}
    max_sim=max(sims.values())
    if max_sim > 0.65:
        correct += 1
        
print("Known acceptance:",correct/len(known_embeddings))

#Unknown rejection
reject=0
for emb in unknown_embeddings:
    sims={user:cosine_similarity(emb,gallery[user]) for user in gallery}
    max_sim= max(sims.values())
    if max_sim<0.65:
        reject += 1

print("Unknown rejection :",reject/len(unknown_embeddings))

Known acceptance: 0.58125
Unknown rejection : 0.95625


In [19]:
#Önce hepsini kapat
for layer in base_model.layers:
    layer.trainable = False
    
#Son 3 katmanı aç(Conv256,BN,Dense)
for layer in base_model.layers[-3:]:
    layer.trainable = True
    
base_model.summary()


Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 300, 90)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 300, 64)        │        28,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 300, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 150, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 150, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 150, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 75, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 75, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 75, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 186,816 (729.75 KB)

 Trainable params: 33,408 (130.50 KB)

 Non-trainable params: 153,408 (599.25 KB)

In [20]:
#Learnin rate düşür
optimizer =tf.keras.optimizers.Adam(learning_rate=1e-5)

In [21]:
EPOCHS = 15

In [22]:
for epoch in range(EPOCHS):
    epoch_loss = 0
    
    for step in range(STEPS_PER_EPOCH):
        a, p, n = generate_triplet_batch(X_train, y_train, BATCH_SIZE)
        loss = train_step(a, p, n)
        epoch_loss += loss.numpy()
        
    print(f"Epoch {epoch+1}, Loss: {epoch_loss/STEPS_PER_EPOCH:.4f}")

Epoch 1, Loss: 12.0248
Epoch 2, Loss: 12.4122
Epoch 3, Loss: 12.8605
Epoch 4, Loss: 11.2507
Epoch 5, Loss: 11.6915
Epoch 6, Loss: 10.5319
Epoch 7, Loss: 9.7773
Epoch 8, Loss: 12.2204
Epoch 9, Loss: 11.0845
Epoch 10, Loss: 10.3177
Epoch 11, Loss: 10.5971
Epoch 12, Loss: 10.4694
Epoch 13, Loss: 10.6285
Epoch 14, Loss: 11.1489
Epoch 15, Loss: 8.9493


In [23]:
#Katmanları tek tek ayır

for layer in base_model.layers:
    if "con1d_2" in layer.name:
        layer.trainable = True
    elif "dense" in layer.name:
        layer.trainable = True
    else:
        layer.trainable = False
        
base_model.summary()

Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 300, 90)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 300, 64)        │        28,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 300, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 150, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 150, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 150, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 75, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 75, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 75, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 186,816 (729.75 KB)

 Trainable params: 32,896 (128.50 KB)

 Non-trainable params: 153,920 (601.25 KB)

In [24]:
#LB biraz arttıralım

optimizer = tf.keras.optimizers.Adam(learning_rate=3e-5)

In [25]:
for epoch in range(EPOCHS):
    epoch_loss = 0
    
    for step in range(STEPS_PER_EPOCH):
        a, p, n = generate_triplet_batch(X_train, y_train, BATCH_SIZE)
        loss = train_step(a, p, n)
        epoch_loss += loss.numpy()
        
    print(f"Epoch {epoch+1}, Loss: {epoch_loss/STEPS_PER_EPOCH:.4f}")

Epoch 1, Loss: 8.9460
Epoch 2, Loss: 8.6679
Epoch 3, Loss: 9.4781
Epoch 4, Loss: 9.6775
Epoch 5, Loss: 8.4390
Epoch 6, Loss: 8.9330
Epoch 7, Loss: 7.9025
Epoch 8, Loss: 7.2227
Epoch 9, Loss: 8.5083
Epoch 10, Loss: 6.8453
Epoch 11, Loss: 7.3054
Epoch 12, Loss: 8.3597
Epoch 13, Loss: 7.6367
Epoch 14, Loss: 6.3602
Epoch 15, Loss: 7.0901


In [26]:
#Train embeddingi yeniden yapalım

train_embeddings = base_model.predict(X_train,verbose=0)
train_embeddings = normalize(train_embeddings,axis=1)

In [27]:
#Gallery yeniden oluşturuluyor

gallery={}

for user in train_users:
    user_mask =(y_train == user)
    user_embeddings = train_embeddings[user_mask]
    
    mean_embedding = np.mean(user_embeddings ,axis=0)
    mean_embedding = mean_embedding / np.linalg.norm(mean_embedding)
    
    gallery[user]=mean_embedding
    
print("Gallery hazır.")

Gallery hazır.


In [28]:
# Known /unknown embedding çıkar

known_embeddings = base_model.predict(X_known,verbose=0)
unknown_embeddings = base_model.predict(X_unknown,verbose=0)

known_embeddings = normalize(known_embeddings,axis=1)
unknown_embeddings = normalize(unknown_embeddings,axis=1)

In [29]:
#open-set test tekrar 

#known acceptance

correct = 0
for emb in known_embeddings:
    sims = {user:np.dot(emb,gallery[user]) for user in gallery}
    max_sim = max(sims.values())
    if max_sim > 0.65:
        correct += 1
        
print("Known acceptance:",correct /len(known_embeddings))

#Unknown rejection

reject = 0
for emb in unknown_embeddings:
    sims = {user:np.dot(emb,gallery[user]) for user in gallery}
    max_sim = max(sims.values())
    if max_sim < 0.65:
        reject += 1
        
print("Unknown rejection:",reject /len(unknown_embeddings))


Known acceptance: 0.5833333333333334
Unknown rejection: 0.9375


In [30]:
# Train embeddings
train_embeddings = base_model.predict(X_train, verbose=0)
train_embeddings = normalize(train_embeddings, axis=1)

# Known test embeddings
known_embeddings = base_model.predict(X_known, verbose=0)
known_embeddings = normalize(known_embeddings, axis=1)

# Unknown test embeddings
unknown_embeddings = base_model.predict(X_unknown, verbose=0)
unknown_embeddings = normalize(unknown_embeddings, axis=1)

print("Embedding çıkarıldı.")

Embedding çıkarıldı.


In [31]:
gallery = {}

for user in train_users:
    user_mask = (y_train == user)
    user_embeddings = train_embeddings[user_mask]

    mean_embedding = np.mean(user_embeddings, axis=0)
    mean_embedding = mean_embedding / np.linalg.norm(mean_embedding)

    gallery[user] = mean_embedding

print("Gallery oluşturuldu.")
print("Gallery users:", gallery.keys())

Gallery oluşturuldu.
Gallery users: dict_keys([0, 1, 2, 3, 6, 7])


In [32]:
# USER 8 enrollment

user8_mask = (y_user == 8)
X_user8 = X[user8_mask]

X_user8 = (X_user8 - mean) / std  # aynı normalizasyonu uygula

user8_embeddings = base_model.predict(X_user8, verbose=0)
user8_embeddings = normalize(user8_embeddings, axis=1)

user8_mean = np.mean(user8_embeddings, axis=0)
user8_mean = user8_mean / np.linalg.norm(user8_mean)

gallery[8] = user8_mean

print("User 8 sisteme eklendi.")

User 8 sisteme eklendi.


In [33]:
correct = 0

for emb in user8_embeddings:
    sims = {user: np.dot(emb, gallery[user]) for user in gallery}
    pred_user = max(sims, key=sims.get)

    if pred_user == 8:
        correct += 1

print("User 8 identification accuracy:", correct / len(user8_embeddings))

User 8 identification accuracy: 0.7375


In [34]:
base_model.save("../models/embedding_model.h5")
print("Embedding model kaydedildi.")

Embedding model kaydedildi.


In [35]:
def generate_hard_triplet_batch(X, y, model, batch_size=32):
    anchors = []
    positives = []
    negatives = []

    unique_users = np.unique(y)

    # Önce tüm embedding'leri çıkar
    all_embeddings = model.predict(X, verbose=0)
    all_embeddings = normalize(all_embeddings, axis=1)

    for _ in range(batch_size):
        user = np.random.choice(unique_users)
        user_indices = np.where(y == user)[0]

        # Anchor & Positive
        a_idx, p_idx = np.random.choice(user_indices, 2, replace=False)

        anchor_emb = all_embeddings[a_idx]

        # Negatif adayları
        neg_indices = np.where(y != user)[0]
        neg_embeddings = all_embeddings[neg_indices]

        # Anchor'a en yakın negative'i bul
        sims = np.dot(neg_embeddings, anchor_emb)
        hardest_neg_idx = neg_indices[np.argmax(sims)]

        anchors.append(X[a_idx])
        positives.append(X[p_idx])
        negatives.append(X[hardest_neg_idx])

    return np.array(anchors), np.array(positives), np.array(negatives)

In [36]:
for epoch in range(EPOCHS):
    epoch_loss = 0

    for step in range(STEPS_PER_EPOCH):
        a, p, n = generate_hard_triplet_batch(X_train, y_train, base_model, BATCH_SIZE)
        loss = train_step(a, p, n)
        epoch_loss += loss.numpy()

    print(f"Epoch {epoch+1}, Loss: {epoch_loss/STEPS_PER_EPOCH:.4f}")

Epoch 1, Loss: 124.6853
Epoch 2, Loss: 114.4659
Epoch 3, Loss: 102.8963
Epoch 4, Loss: 96.8710
Epoch 5, Loss: 85.3431
Epoch 6, Loss: 81.5300
Epoch 7, Loss: 76.8587
Epoch 8, Loss: 70.9329
Epoch 9, Loss: 63.9728
Epoch 10, Loss: 62.7272
Epoch 11, Loss: 56.6231
Epoch 12, Loss: 54.5476
Epoch 13, Loss: 50.4597
Epoch 14, Loss: 50.2594
Epoch 15, Loss: 47.9569


In [38]:
base_model.save("../models/embedding_model.h5")
print("Yeni embedding modeli kaydedildi.")

Yeni embedding modeli kaydedildi.
